# Simulador 14.2 — Programas Concurrentes RV-IV
### *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas · Facultad de Psicología, UNAM**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arbouria/Libro-ACA-2026/blob/main/Simuladores/sim14_3_rv_iv.ipynb)


In [1]:
# Instalación (solo necesaria en Colab)
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'],
                   check=False)


In [2]:
HEADER = "<div style='background:#EBF4FF; border-left:5px solid #2C5282; padding:20px 24px; margin-bottom:20px; font-family:Georgia,Palatino,serif; border-radius:0 4px 4px 0; border:1px solid #BEE3F8;'><h3 style='color:#2C5282; margin:0 0 10px 0; font-size:1.1em;'>Simulador 14.3 &mdash; Programas Concurrentes RV-IV: &iquest;Rentabilidad o Maximizaci&oacute;n?</h3><p style='margin:0 0 9px 0; color:#2D3748; font-size:0.95em; line-height:1.65;'>Dos organismos aprenden en el mismo programa concurrente <strong>raz&oacute;n variable (RV) &mdash; intervalo variable (IV)</strong>. El <strong style='color:#2C5282;'>organismo de rentabilidad</strong> (azul) aplica melioration: compara las tasas locales de refuerzo de las dos opciones y desplaza su comportamiento hacia la m&aacute;s rentable hasta que se igualan. El <strong style='color:#C05621;'>organismo de maximizaci&oacute;n</strong> (naranja) aplica ascenso de colina sobre el total de refuerzos: incrementa la fracci&oacute;n en RV porque eso siempre aumenta la ganancia global en este entorno.</p><p style='margin:0; color:#2D3748; font-size:0.95em; line-height:1.65;'><strong>Panel 1:</strong> distribuci&oacute;n de respuestas. &nbsp;<strong>Panel 2:</strong> rentabilidades locales del organismo de igualaci&oacute;n. &nbsp;<strong>Panel 3:</strong> refuerzos acumulados y costo de igualar.</p></div>"

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

AZUL    = '#2C5282'
NARANJA = '#C05621'
VERDE   = '#276749'
GRIS    = '#718096'
FONDO   = '#F7FAFC'

display(HTML(HEADER))

# ===================================================================
# SIMULACION
# ===================================================================

def simular(n_ensayos, n_rv, t_iv, alpha_mel=0.15, alpha_max=0.025,
            batch=50, seed=42):
    '''
    Simula dos organismos en un programa concurrente RV-IV.
    Modelo del IV: acumulacion libre.
      vi_acc crece en 1/t_iv por ensayo. Cuando el organismo visita IV:
          P(refuerzo) = min(vi_acc, 1.0)  y  vi_acc <- 0.
      Total refuerzos IV aprox n_ensayos/t_iv, constante en la distribucion.
    Org. 1 (Rentabilidad): melioration; equilibrio teorico p_rv = 1 - n_rv/t_iv.
    Org. 2 (Maximizacion): incrementa p_rv cada batch (gradiente positivo siempre).
    '''
    rng  = np.random.default_rng(seed)
    fill = 1.0 / t_iv
    p1 = 0.5;  vi_acc1 = 0.0
    rv_r1 = rv_n1 = vi_r1 = vi_n1 = 0
    _lr_rv = 1.0 / n_rv;  _lr_iv = 1.0 / n_rv
    p2 = 0.5;  vi_acc2 = 0.0
    p1_h  = np.zeros(n_ensayos);  p2_h  = np.zeros(n_ensayos)
    lrv_h = np.full(n_ensayos, 1.0 / n_rv)
    lri_h = np.full(n_ensayos, 1.0 / n_rv)
    cum1  = np.zeros(n_ensayos);  cum2  = np.zeros(n_ensayos)
    tot1  = tot2 = 0
    for t in range(n_ensayos):
        vi_acc1 = min(vi_acc1 + fill, 3.0)
        vi_acc2 = min(vi_acc2 + fill, 3.0)
        c1 = 0 if rng.random() < p1 else 1
        r1 = 0
        if c1 == 0:
            rv_n1 += 1
            if rng.random() < 1.0 / n_rv:
                r1 = 1;  rv_r1 += 1
        else:
            vi_n1 += 1
            if rng.random() < min(vi_acc1, 1.0):
                r1 = 1;  vi_r1 += 1
            vi_acc1 = 0.0
        tot1 += r1;  cum1[t] = tot1
        p1_h[t] = p1;  lrv_h[t] = _lr_rv;  lri_h[t] = _lr_iv
        if (t + 1) % batch == 0:
            if rv_n1 >= 5 and vi_n1 >= 5:
                _lr_rv = rv_r1 / rv_n1
                _lr_iv = vi_r1 / vi_n1
                p1 = float(np.clip(
                    p1 + alpha_mel * (_lr_rv - _lr_iv), 0.05, 0.95))
            rv_r1 = rv_n1 = vi_r1 = vi_n1 = 0
        c2 = 0 if rng.random() < p2 else 1
        r2 = 0
        if c2 == 0:
            if rng.random() < 1.0 / n_rv:
                r2 = 1
        else:
            if rng.random() < min(vi_acc2, 1.0):
                r2 = 1
            vi_acc2 = 0.0
        tot2 += r2;  cum2[t] = tot2;  p2_h[t] = p2
        if (t + 1) % batch == 0:
            p2 = float(np.clip(p2 + alpha_max, 0.05, 0.97))
    def smooth(arr, w):
        cs = np.cumsum(arr)
        cs[w:] = cs[w:] - cs[:-w]
        return cs / np.minimum(np.arange(1, len(arr) + 1), w)
    f_iv_eq = float(np.clip(n_rv / t_iv, 0.05, 0.95))
    return dict(
        p1=smooth(p1_h, 250), p2=smooth(p2_h, 250),
        lr_rv=smooth(lrv_h, 400), lr_iv=smooth(lri_h, 400),
        cum1=cum1, cum2=cum2, p_rv_eq=1.0 - f_iv_eq,
        final_p1=float(np.mean(p1_h[-300:])),
        final_p2=float(np.mean(p2_h[-300:])),
        total1=tot1, total2=tot2, n_rv=n_rv, t_iv=t_iv, n=n_ensayos)

# ===================================================================
# VISUALIZACION
# ===================================================================

def graficar(res):
    n = res['n'];  x = np.arange(n)
    eq = res['p_rv_eq'];  n_rv = res['n_rv'];  t_iv = res['t_iv']
    fig, axes = plt.subplots(3, 1, figsize=(12, 11))
    fig.patch.set_facecolor('white')
    fig.subplots_adjust(hspace=0.50, left=0.09, right=0.97, top=0.92, bottom=0.06)
    ax = axes[0]
    ax.plot(x, res['p2'], color=NARANJA, lw=1.9, alpha=0.92,
            label='Maximizacion  (RV final: {:.0%})'.format(res['final_p2']))
    ax.plot(x, res['p1'], color=AZUL, lw=1.9, alpha=0.92,
            label='Rentabilidad  (RV final: {:.0%})'.format(res['final_p1']))
    ax.axhline(eq, color=AZUL, lw=1.1, ls='--', alpha=0.45,
               label='Equilibrio teorico igualacion: RV {:.0%}'.format(eq))
    ax.set_ylim(0.30, 1.02)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_xlabel('Respuesta', fontsize=10)
    ax.set_ylabel('% respuestas en RV', fontsize=10)
    ax.set_title('Distribucion de Respuestas', fontsize=11,
                 fontweight='bold', color='#2D3748', pad=7)
    ax.legend(fontsize=9, loc='lower right', framealpha=0.93)
    ax.set_facecolor(FONDO);  ax.grid(True, alpha=0.25)
    ax.axhspan(res['final_p1']-0.03, res['final_p1']+0.03,
               color=AZUL, alpha=0.07, zorder=0)
    ax = axes[1]
    s = 300
    ax.plot(x[s:], res['lr_rv'][s:], color=AZUL,  lw=1.9, label='Rentabilidad local RV')
    ax.plot(x[s:], res['lr_iv'][s:], color=VERDE, lw=1.9, label='Rentabilidad local IV')
    ax.axhline(1.0/n_rv, color=GRIS, lw=1.0, ls=':', alpha=0.65,
               label='Valor teorico: 1/{} = {:.3f}'.format(n_rv, 1/n_rv))
    ax.set_ylim(0, min(1.0/n_rv * 3.5, 1.05))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.set_xlabel('Respuesta', fontsize=10)
    ax.set_ylabel('Refuerzos / respuesta', fontsize=10)
    ax.set_title('Rentabilidades Locales - Organismo de Igualacion',
                 fontsize=11, fontweight='bold', color='#2D3748', pad=7)
    ax.legend(fontsize=9, framealpha=0.93)
    ax.set_facecolor(FONDO);  ax.grid(True, alpha=0.25)
    ax = axes[2]
    ax.plot(x, res['cum2'], color=NARANJA, lw=1.9, alpha=0.92,
            label='Maximizacion  (total: {:,})'.format(res['total2']))
    ax.plot(x, res['cum1'], color=AZUL, lw=1.9, alpha=0.92,
            label='Rentabilidad  (total: {:,})'.format(res['total1']))
    ax.fill_between(x, res['cum1'], res['cum2'],
                    where=(res['cum2'] >= res['cum1']),
                    color=NARANJA, alpha=0.12, zorder=0)
    costo = res['total2'] - res['total1']
    pct   = 100.0 * costo / max(res['total2'], 1)
    ax.text(0.97, 0.09,
            'Costo de igualar: {:+d} refuerzos ({:.0f}%)'.format(costo, pct),
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9.5,
            color='#7B341E',
            bbox=dict(boxstyle='round,pad=0.45', facecolor='#FFFAF0',
                      edgecolor='#C05621', alpha=0.92))
    ax.set_xlabel('Respuesta', fontsize=10)
    ax.set_ylabel('Refuerzos acumulados', fontsize=10)
    ax.set_title('Refuerzos Acumulados en la Sesion',
                 fontsize=11, fontweight='bold', color='#2D3748', pad=7)
    ax.legend(fontsize=9, loc='upper left', framealpha=0.93)
    ax.set_facecolor(FONDO);  ax.grid(True, alpha=0.25)
    fig.suptitle(
        'RV {} - IV {} resp. media  |  {:,} respuestas  |  '
        'Equilibrio igualacion: RV {:.0%} / IV {:.0%}'.format(
            n_rv, t_iv, n, eq, 1-eq),
        fontsize=10.5, color='#2D3748', fontweight='bold', y=0.975)
    plt.show()

# ===================================================================
# CONTROLES
# ===================================================================

display(HTML("<hr style='border:none; border-top:1px solid #E2E8F0; margin:18px 0 14px;'>"))
S = {'description_width': '145px'}
L = widgets.Layout(width='430px')
w_nrv = widgets.IntSlider(value=10, min=3,  max=30,   step=1,
    description='RV (razon media n):', style=S, layout=L)
w_tiv = widgets.IntSlider(value=30, min=10, max=120,  step=5,
    description='IV (intervalo, resp.):', style=S, layout=L)
w_ntr = widgets.IntSlider(value=3000, min=600, max=5000, step=200,
    description='Total respuestas:', style=S, layout=L)
w_alp = widgets.FloatSlider(value=0.15, min=0.02, max=0.30, step=0.02,
    description='alfa (melioration):', style=S, layout=L, readout_format='.2f')
row1 = widgets.HBox([w_nrv, w_tiv])
row2 = widgets.HBox([w_ntr, w_alp])
info_out = widgets.Output()
def actualizar_info(change=None):
    f_iv = w_nrv.value / w_tiv.value;  p_rv = 1 - f_iv
    with info_out:
        info_out.clear_output(wait=True)
        display(HTML(
            "<p style='font-family:monospace; font-size:0.90em; color:#2D3748; "
            "background:#F0FFF4; border-left:3px solid #276749; padding:8px 12px;'>"
            "Equilibrio teorico igualacion: "
            "f_IV = {}/{} = {:.3f}  ->  RV {:.0%} / IV {:.0%}"
            "</p>".format(w_nrv.value, w_tiv.value, f_iv, p_rv, f_iv)))
w_nrv.observe(actualizar_info, names='value')
w_tiv.observe(actualizar_info, names='value')
actualizar_info()
btn = widgets.Button(description='Correr simulacion', button_style='primary',
                     layout=widgets.Layout(width='200px', height='38px'))
out = widgets.Output()
display(row1, row2, info_out, btn, out)
def al_hacer_clic(b):
    out.clear_output(wait=True)
    with out:
        graficar(simular(w_ntr.value, w_nrv.value, w_tiv.value, w_alp.value))
btn.on_click(al_hacer_clic)
with out:
    graficar(simular(3000, 10, 30, 0.15))


Output()

Button(button_style='primary', description='Correr simulacion', layout=Layout(height='38px', width='200px'), s…

Output()

---

## Ejercicios

**Ejercicio 1 (básico).**  
Con los parámetros por defecto (RV 10, IV 30 respuestas), ¿a qué porcentaje de RV converge el organismo de igualación? ¿Y el de maximización? Compara el primero con el equilibrio teórico (f_IV = n_RV / t_IV) de la nota verde. Lee el costo de igualar del panel 3 y anótalo.

---

**Ejercicio 2 (intermedio).**  
Antes de correr: si cambias el IV de 30 a 60 respuestas, ¿qué debería pasarle al equilibrio de igualación? Corre la simulación. ¿Cambia el costo de igualar? ¿Por qué?

---

**Ejercicio 3 (intermedio).**  
Establece n_RV = 30 (RV más pobre). ¿Para qué valor de n_RV las predicciones de igualación y maximización serían indistinguibles? ¿Qué dice eso sobre el poder discriminativo del diseño RV-IV?

---

**Ejercicio 4 (avanzado).**  
Observa el panel de rentabilidades en las primeras 500 respuestas. ¿Cuál opción es más rentable mientras el organismo está cerca de 50/50? ¿Es coherente con la dirección del ajuste de melioration?

---

**Ejercicio 5 (avanzado).**  
Aumenta alfa a 0.28. ¿Converge limpiamente o hay oscilaciones en el panel de rentabilidades? ¿Qué dice esto sobre la relación entre velocidad de ajuste y precisión del equilibrio?
